<a href="https://colab.research.google.com/github/TahaNawazkhan/Artificial-intelligence-/blob/main/lab%202%20task%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}


def is_valid(state):
    """A valid 8-puzzle state is a permutation of 0..8."""
    return len(state) == 9 and sorted(state) == list(range(9))


def inversions(state):
    tiles = [x for x in state if x != 0]
    return sum(1 for i in range(len(tiles)) for j in range(i + 1, len(tiles)) if tiles[i] > tiles[j])


def is_solvable(start, goal=GOAL):
    """On a 3x3 board, a state can reach the goal only if both have the same inversion parity."""
    return inversions(start) % 2 == inversions(goal) % 2


def apply_move(state, move):
    b = state.index(0)
    r, c = divmod(b, 3)
    dr, dc = MOVES[move]
    n = (r + dr) * 3 + (c + dc)
    s = list(state)
    s[b], s[n] = s[n], s[b]
    return tuple(s)


def manhattan(state, goal):
    pos = {v: i for i, v in enumerate(goal)}
    return sum(abs(i // 3 - pos[v] // 3) + abs(i % 3 - pos[v] % 3)
               for i, v in enumerate(state) if v != 0)


def astar(start, goal):
    """A* with Manhattan distance. Returns (path, expanded, generated)."""
    expanded, generated, tie = 0, 1, 0
    frontier = [(manhattan(start, goal), 0, tie, start)]
    parent = {start: None}
    best_g = {start: 0}
    while frontier:
        f, g, _, state = heapq.heappop(frontier)
        if g > best_g[state]:
            continue
        if state == goal:
            path = []
            while parent[state] is not None:
                state, move = parent[state]
                path.append(move)
            return path[::-1], expanded, generated
        expanded += 1
        r, c = divmod(state.index(0), 3)
        for move, (dr, dc) in MOVES.items():
            if 0 <= r + dr < 3 and 0 <= c + dc < 3:
                nxt = apply_move(state, move)
                generated += 1
                ng = g + 1
                if ng < best_g.get(nxt, float("inf")):
                    best_g[nxt] = ng
                    parent[nxt] = (state, move)
                    tie += 1
                    heapq.heappush(frontier, (ng + manhattan(nxt, goal), ng, tie, nxt))


def solve_puzzle(start, goal=GOAL):
    """Check the state first; only search if it is valid and solvable."""
    if not is_valid(start) or not is_valid(goal):
        print(f"{start}: invalid state (must be a permutation of 0-8).")
        return None
    if not is_solvable(start, goal):
        print(f"{start}: UNSOLVABLE (inversions = {inversions(start)}, "
              f"parity differs from the goal). No search was started.")
        return None
    t0 = time.perf_counter()
    path, expanded, generated = astar(start, goal)
    secs = time.perf_counter() - t0
    print(f"{start}: solvable. Length = {len(path)}, expanded = {expanded}, "
          f"generated = {generated}, time = {secs:.4f}s")
    return path


if __name__ == "__main__":
    tests = [
        (1, 2, 3, 4, 5, 6, 8, 7, 0),   # unsolvable (last two tiles swapped)
        (8, 1, 3, 4, 0, 2, 7, 6, 5),   # solvable
        (1, 2, 3, 4, 5, 6, 7, 8, 0),   # already the goal
        (1, 1, 3, 4, 5, 6, 7, 8, 0),   # invalid
    ]
    for t in tests:
        solve_puzzle(t)

    # Proof that search is never started for the unsolvable state:
    searched = []
    _real = astar
    astar = lambda *a, **k: searched.append(a) or _real(*a, **k)
    solve_puzzle((1, 2, 3, 4, 5, 6, 8, 7, 0))
    print("\nSearch called for the unsolvable state?", "YES" if searched else "NO")

(1, 2, 3, 4, 5, 6, 8, 7, 0): UNSOLVABLE (inversions = 1, parity differs from the goal). No search was started.
(8, 1, 3, 4, 0, 2, 7, 6, 5): solvable. Length = 14, expanded = 76, generated = 207, time = 0.0009s
(1, 2, 3, 4, 5, 6, 7, 8, 0): solvable. Length = 0, expanded = 0, generated = 1, time = 0.0000s
(1, 1, 3, 4, 5, 6, 7, 8, 0): invalid state (must be a permutation of 0-8).
(1, 2, 3, 4, 5, 6, 8, 7, 0): UNSOLVABLE (inversions = 1, parity differs from the goal). No search was started.

Search called for the unsolvable state? NO
